# External independent reproduction guide

This notebook checks released inputs and rebuilds the figures for **Can genes be described in natural language like images?** The manuscript contains five main figures and seven Extended Data figures.

The notebook reads released assets only. It does not download or regenerate public RNA-seq data from the original archives.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

CODE_REPO = Path(os.environ.get("RNA_PORTRAIT_CODE_REPO", Path.cwd())).resolve()
DATA_PACKAGE = Path(os.environ.get(
    "RNA_PORTRAIT_DATA_PACKAGE",
    CODE_REPO.parent / "released_data_package",
)).resolve()
OUTPUT_DIR = Path(os.environ.get(
    "RNA_PORTRAIT_NOTEBOOK_OUTPUT",
    CODE_REPO / "outputs" / "manuscript_figures",
)).resolve()

assert (CODE_REPO / "workflows" / "figures" / "current_submission").is_dir(), CODE_REPO
assert (DATA_PACKAGE / "source_data").is_dir(), DATA_PACKAGE
assert (DATA_PACKAGE / "figure_panel_sources").is_dir(), DATA_PACKAGE
print("code repository:", CODE_REPO)
print("data package:", DATA_PACKAGE)
print("figure output:", OUTPUT_DIR)

## 1. Release-integrity checks

Every CSV in `source_data/` must appear exactly once in the manifest. The released set contains the worked-example, calibration, reliability and disease-composition tables required by the manuscript.

In [ ]:
source_dir = DATA_PACKAGE / "source_data"
manifest_path = source_dir / "source_data_manifest.csv"
manifest = pd.read_csv(manifest_path)
listed = [Path(value).name for value in manifest["draft_copy"]]
actual = sorted(
    path.name for path in source_dir.glob("*.csv")
    if path.name != manifest_path.name and not path.name.startswith("._")
)
assert len(listed) == len(set(listed))
assert sorted(listed) == actual
print(f"source-data manifest: {len(manifest)} entries, complete")
manifest[["name", "description"]].head(8)

In [ ]:
required_training = {
    "expr_log.parquet", "meta.csv", "meta.parquet",
    "genes.npy", "sample_ids.npy", "summary.json",
}
training_dir = DATA_PACKAGE / "processed_data" / "training_matrix"
missing_training = sorted(required_training - {path.name for path in training_dir.iterdir()})
assert not missing_training, missing_training

required_panel_sources = {
    "S1a_training_embedding_pca_by_split.svg",
    "S1b_training_embedding_pca_by_site.svg",
    "S1c_training_embedding_pca_by_tumour_status.svg",
    "S1d_training_embedding_pca_by_disease_label.svg",
    "S1e_training_embedding_pca_by_source.svg",
    "S1f_training_embedding_pca_by_rna_text_gap.svg",
    "S3a_cosine_control_distributions.svg",
    "S4a_internal_reliability_curve.svg",
    "S4b_internal_risk_coverage_curve.svg",
    "S8c_within_label_evidence_separation.svg",
    "S9b_quality_metrics_by_portrait.svg",
    "S10b_reliability_tiers_by_portrait.svg",
    "S10c_boundary_flags_by_portrait_heatmap.svg",
    "Extended_Data_Fig_4_pre_x_label_wrap_20260812.svg",
}
panel_dir = DATA_PACKAGE / "figure_panel_sources"
missing_panels = sorted(required_panel_sources - {path.name for path in panel_dir.iterdir()})
assert not missing_panels, missing_panels
print("training matrix and 14 released vector-panel sources: present")

## 2. Current manuscript figure map

In [ ]:
figure_map = pd.DataFrame([
    ("Fig. 1", "Figure_1_architecture", "RNA–language architecture"),
    ("Fig. 2", "Figure_2_RNA_language_alignment", "RNA–language alignment"),
    ("Fig. 3", "Figure_3_biological_grounding", "Molecular grounding and worked examples"),
    ("Fig. 4", "Figure_4_portraits_not_single_labels", "Predefined-label heterogeneity"),
    ("Fig. 5", "Figure_5_stress_tests_and_reliability", "Mixtures, source controls and reliability"),
    *[(f"Extended Data Fig. {i}", f"Extended_Data_Fig_{i}", "Current Extended Data plate") for i in range(1, 8)],
], columns=["manuscript_number", "output_stem", "content"])
figure_map

## 3. Rebuild all 5+7 figures

The current Python/Matplotlib workflow writes SVG, PDF, PNG and RGB/LZW TIFF. Set `RUN_FIGURE_REBUILD = False` only when inspecting an already generated output directory.

In [ ]:
RUN_FIGURE_REBUILD = True

if RUN_FIGURE_REBUILD:
    command = [
        sys.executable,
        str(CODE_REPO / "workflows" / "figures" / "current_submission" / "make_submission_figures.py"),
        "--data-package", str(DATA_PACKAGE),
        "--output-dir", str(OUTPUT_DIR),
    ]
    subprocess.run(command, check=True)
print("figure workflow complete")

In [ ]:
formats = ["svg", "pdf", "png", "tiff"]
expected = {
    f"{stem}.{extension}"
    for stem in figure_map["output_stem"]
    for extension in formats
}
actual_outputs = {
    path.name for path in OUTPUT_DIR.iterdir()
    if path.is_file() and not path.name.startswith("._") and path.suffix.lstrip(".").lower() in formats
}
assert actual_outputs == expected, {
    "missing": sorted(expected - actual_outputs),
    "unexpected": sorted(actual_outputs - expected),
}
for stem in figure_map["output_stem"]:
    with Image.open(OUTPUT_DIR / f"{stem}.tiff") as image:
        assert image.mode == "RGB"
        assert image.info.get("compression") == "tiff_lzw"
print("verified: 12 figures × 4 formats; TIFF files are RGB/LZW")

## 4. Quantitative spot checks

In [ ]:
cosine = pd.read_csv(source_dir / "figure_2_cosine_summary.csv")
statements = pd.read_csv(source_dir / "figure_3_statement_support.csv")
thresholds = pd.read_csv(source_dir / "figure_4_thresholds.csv")
mixing = pd.read_csv(source_dir / "figure_5_mixing.csv")
reliability = pd.read_csv(source_dir / "figure_5_reliability_flags.csv")

print("Fig. 2 cosine-summary columns:", list(cosine.columns))
print("Fig. 3 statement rows:", len(statements))
print("Fig. 4 threshold rows:", len(thresholds))
print("Fig. 5 mixing rows:", len(mixing))
print("Fig. 5 reliability flags:", reliability["boundary_label"].tolist())

## 5. Visual inspection sheet

In [ ]:
names = figure_map["output_stem"].tolist()
fig, axes = plt.subplots(4, 3, figsize=(15, 18))
for ax, name in zip(axes.flat, names):
    with Image.open(OUTPUT_DIR / f"{name}.png") as image:
        image.thumbnail((900, 900))
        ax.imshow(image)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
fig.tight_layout()
plt.show()

## Reproduction boundary

The figure route above reproduces the submitted 5+7 figure set from the deposited source-data snapshot and released vector-panel sources. Full model retraining begins from the prepared training matrix and may produce numerically different weights across hardware and software environments. Raw public RNA-seq data remain in their original repositories; accession mapping is provided in `data_availability/public_dataset_manifest.csv`.